### Make a temporary space in /flash

In [1]:
import os, random, string

In [2]:
cwd = os.path.basename(os.getcwd())
tmpdir = os.path.join("/flash/ReiterU/tmp/", 'tomo_' + ''.join(random.choices(string.ascii_uppercase, k=12)))
print(tmpdir)
if not os.path.exists(tmpdir):
    os.makedirs(tmpdir)

/flash/ReiterU/tmp/tomo_WGFQPPCKPAUP


### PARAMETERS

In [3]:
#### path to relevant files
rootdir = "/bucket/ReiterU/OLdev/experiments/20250520_squid_cal520"
recname = "rec5"
savedir = "/bucket/ReiterU/OLdev/analysis/20250528_IGL_monitor_flip_analysis/20250520_rec5"

templatefile = os.path.join(rootdir, "2P", recname, "run2/mcorr_nonrigid/QC/QC_nonrigid/template.tif")
roifile = os.path.join(rootdir, "2P", recname, "ROI_van_renamed.zip")
dfffile = os.path.join(rootdir, "2P", recname, "dff_ROI_van/DFF.hdf5")
syncfile = os.path.join(rootdir, "sync", f"{recname}.tdms")

QI_THRESH = 0.25

### Import stuff

In [4]:
import numpy as np
import pandas as pd
from matplotlib import pyplot as plt
from scipy import signal
import h5py
import subprocess

In [5]:
import sys
sys.path.append("../../../")
from calfuncs.io import read_sync_ttl
from calfuncs.full_field_step import get_ett2, plot_chirp_eta, compute_qi

In [6]:
from matplotlib import font_manager
import matplotlib as mpl

font_manager.fontManager.addfont('/bucket/ReiterU/common/fonts/arial.ttf')
mpl.rcParams.update({
    'font.size': 7,
    'font.family': 'sans-serif',
    'font.sans-serif': "Arial",
    'pdf.fonttype': 42,
    'ps.fonttype': 42,
    'axes.linewidth': 0.5,
    'xtick.major.width': 0.5,
    'xtick.major.size': 2,
    'xtick.minor.size': 2,
    'xtick.minor.width': 0.5,
    'ytick.major.width': 0.5,
    'ytick.minor.width': 0.5,
    'ytick.major.size': 2,
    'ytick.minor.size': 2,
})

### Load TDMS file

In [7]:
sync_ttl = read_sync_ttl(syncfile)
twop_ttl = sync_ttl["twop_ttl"]
twop_re = sync_ttl["twop_re"]
vis_re = sync_ttl["vis_re"]
print(twop_ttl.shape, twop_re.shape, vis_re.shape)

(716500,) (10908,) (60,)


### Load DF/F

In [8]:
with h5py.File(dfffile, "r") as hf:
    F = hf["F"][:]
    dff = hf["dff"][:]
print(F.shape) # F is sampled at 1 kH

(186, 716500)


### Chirp

In [9]:
E = []
for nid in np.arange(0, F.shape[0]):
    ett1 = get_ett2(F, nid, vis_re[0::6], window=[-2000, 19000], baseline_window=[-4000, 70000], subsampling=50)
    ett1["session"] = 0
    ett2 = get_ett2(F, nid, vis_re[2::6], window=[-2000, 19000], baseline_window=[-27000, 43000], subsampling=50)
    ett2["session"] = 1
    ett3 = get_ett2(F, nid, vis_re[4::6], window=[-2000, 19000], baseline_window=[-51000, 19000], subsampling=50)
    ett3["session"] = 2
    ett = pd.concat([ett1, ett2, ett3])
    ett["neuron"] = nid
    ett = ett.set_index(["neuron", "session", "trial", "time"])
    E.append(ett)
E = pd.concat(E)

In [ ]:
dt = 1e-2
stimulus = np.zeros(int(21.0/dt), dtype=np.float32)
stimulus[0:int(2.0/dt)] = 0
stimulus[int(2.0/dt):int(6.0/dt)] = 1
stimulus[int(6.0/dt):int(10.0/dt)] = 0
stimulus[int(10.0/dt):int(12.0/dt)] = 0.5
stimulus[int(12.0/dt):int(20.0/dt)] = 0.5 + 0.5 * signal.chirp(np.linspace(0, 8, int(20.0/dt)-int(12.0/dt)), f0=0.5, f1=10, t1=8, method='logarithmic', phi=90)
stimulus[int(20.0/dt):int(21.0/dt)] = 0

for nid in np.arange(0, dff.shape[0]):
    qi = compute_qi(E, nid)
    print("Neuron:", nid, "QI:", qi)
    savename = os.path.join(tmpdir, f"ETA_{nid:03d}.png")
    plot_chirp_eta(E, nid, stimulus, vmin=-2, vmax=3,  cmap_name="bone_r", figsize=(5,2.0), show_plot=True, figsavename=savename)

### Compute trial-average

In [11]:
R = []
neurons = E.index.get_level_values("neuron").unique()

for n in neurons:
    M = []
    df2 = E.xs(n, level="neuron")
    for i in range(3):
        df3 = df2.xs(i, level="session")
        mat = df3.reset_index().pivot(index="trial", columns="time", values="y").to_numpy()
        M.append(mat)
    M = np.hstack(M)
    M = M[~np.all(np.isnan(M), axis=1)] # remove trials with NaN values
    qi = compute_qi(E, n)
    y_mean = np.nanmean(M, axis=0)
    row = {
        "recording": 0,
        "neuron": n,
        "qi": qi,
    }
    for j, value in enumerate(y_mean):
        row[f"time_{j}"] = value

    R.append(row)

R = pd.DataFrame(R)

In [12]:
display(R)

,recording,neuron,qi,time_0,time_1,time_2,time_3,time_4,time_5,time_6,...,time_1250,time_1251,time_1252,time_1253,time_1254,time_1255,time_1256,time_1257,time_1258,time_1259
0,0,0,0.295325,0.036435,-0.039771,0.132314,0.283739,0.242528,-0.180028,-0.278157,...,0.083816,0.419027,0.334247,0.762542,0.987887,0.896030,0.693016,0.484068,0.578615,0.423477
1,0,1,0.411212,-0.685606,-0.144509,-0.214974,-0.263296,-0.461801,-0.681610,-0.377086,...,-0.284699,-0.349915,0.123126,-0.309486,-0.508572,-0.275283,-0.353907,-0.477002,-0.524793,-0.476718
2,0,2,0.510986,-0.523049,-0.545780,-0.512787,-0.615102,-0.545643,-0.633462,-0.570998,...,0.912821,0.908457,0.916439,0.944522,1.047277,1.395679,1.468494,1.386456,1.103238,1.121169
3,0,3,0.388709,-0.923931,-0.413429,-0.476432,-0.474949,-0.428406,-0.296341,-0.491744,...,-0.042924,0.015651,0.246695,0.234966,0.037995,0.230263,0.447656,0.533680,0.303174,-0.105340
4,0,4,0.434016,1.063382,0.611957,0.634377,0.917108,0.553061,0.460814,0.878728,...,-0.798598,-0.590739,-0.550339,-0.048477,-0.040058,-0.367691,-0.444710,-0.514502,-0.131286,0.085997
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
181,0,181,0.796814,0.229412,0.346954,0.249015,0.158447,0.179857,0.141338,0.207163,...,0.716341,0.957919,1.001413,1.013912,0.996376,0.846286,0.986570,1.016745,1.023221,1.101777
182,0,182,0.176232,-0.757178,-0.433064,0.036593,-0.145954,0.238527,0.074753,-0.179687,...,-0.444125,-0.438628,-0.587714,-0.276381,-0.719426,-0.759366,-0.494578,-0.370530,-0.832063,-0.330233
183,0,183,0.189840,0.286630,0.232486,0.441966,0.566292,-0.135565,0.140479,0.310794,...,-0.554899,-0.814143,-0.739808,-0.008263,0.071081,-0.166202,-0.460970,-0.050465,0.172647,-0.220932
184,0,184,0.222866,0.038945,0.523916,0.147835,0.647837,0.855095,0.427896,0.786109,...,-0.058576,0.063647,0.150369,-0.070308,0.140935,-0.086795,-0.224074,-0.507686,-0.533606,-0.283546


In [13]:
with pd.HDFStore(os.path.join(tmpdir, "response_table.hdf5"), mode='w') as store:
    store.put('E', E)
    store.put('responses', R)

### Copy data to bucket

In [14]:
cmd1 = f'ssh deigo "mkdir -p {savedir}"'
cmd2 = f'ssh deigo "cp -r {tmpdir}/* {savedir}"'
print(cmd1)
print(cmd2)

subprocess.run(cmd1, shell=True, check=True)
subprocess.run(cmd2, shell=True, check=True)

ssh deigo "mkdir -p /bucket/ReiterU/OLdev/analysis/20250528_IGL_monitor_flip_analysis/20250520_rec5"
ssh deigo "cp -r /flash/ReiterU/tmp/tomo_WGFQPPCKPAUP/* /bucket/ReiterU/OLdev/analysis/20250528_IGL_monitor_flip_analysis/20250520_rec5"


Offending key for IP in /home/t/tomoyuki-mano1/.ssh/known_hosts:1
Matching host key in /home/t/tomoyuki-mano1/.ssh/known_hosts:6


CompletedProcess(args='ssh deigo "cp -r /flash/ReiterU/tmp/tomo_WGFQPPCKPAUP/* /bucket/ReiterU/OLdev/analysis/20250528_IGL_monitor_flip_analysis/20250520_rec5"', returncode=0)